# CRIVO — Treino Curricular Longo (GPU)

Este é o treino mais amplo desta sequência. Ele parte do **checkpoint composicional mais recente** salvo no Google Drive e executa **10.000 passos úteis**, em seis fases:

1. identidade, categoria, função e invariância;
2. binding relacional entre entidades e funções;
3. contrastes, negação, hipótese e correção;
4. composição e inferência multi-relação;
5. memória de diálogo, correções e referências contextuais;
6. generalização e geração natural.

Técnicas usadas: **currículo crescente, repetição espaçada/replay, hard negatives com loss de ranking, unfreezing progressivo, warmup + cosine decay, checkpoints por fase e seleção pelo conjunto de desenvolvimento**.

No final, o melhor checkpoint é submetido a uma bateria cega de **500 casos com entidades artificiais que nunca aparecem no treino**. O resultado continua com `ativo_no_chat=false`; passar no treino não promove automaticamente o modelo.

### Uso
Ative uma GPU no Colab e clique em **Executar tudo**. Não precisa editar nenhuma célula.


In [ ]:
!nvidia-smi
!pip -q install 'tokenizers>=0.20,<1'

import torch, pathlib, sys
from google.colab import drive

assert torch.cuda.is_available(), 'Ative GPU em Ambiente de execução > Alterar tipo de ambiente > GPU.'
print('✅ GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')

ROOT=pathlib.Path('/content/drive/MyDrive/CRIVO/checkpoints')
SRC=ROOT/'escola_composicional_real'
assert SRC.exists(), f'Não encontrei {SRC}'
cands=sorted([p for p in SRC.glob('crivo_composicional_*') if (p/'pesos.pt').exists() and (p/'tokenizer.json').exists()])
assert cands, 'Nenhum checkpoint composicional foi encontrado.'
print('✅ checkpoint de entrada:', cands[-1])


In [ ]:
%cd /content
!rm -rf CRIVO
!git clone -q --branch colab/treino-curricular-longo-20261011 --single-branch https://github.com/HROSONE/CRIVO.git
%cd /content/CRIVO
!git rev-parse HEAD
print('✅ Código do treino carregado.')


## Treino real

A próxima célula faz todo o processo: avaliação cega inicial, seis fases de treino, avaliação de desenvolvimento ao fim de cada fase, checkpoints no Drive, escolha da melhor fase e avaliação cega final.

Os arquivos serão salvos em:

`Meu Drive/CRIVO/checkpoints/escola_curricular_longa/`


In [ ]:
%cd /content/CRIVO
!PYTHONHASHSEED=0 python scripts/treino_curricular_longo.py --drive-root /content/drive/MyDrive/CRIVO/checkpoints --batch-size 16 --blind-entities 100


## Resultado

Quando terminar, procure no fim da saída por **TREINO CURRICULAR LONGO FINALIZADO**. O notebook mostrará:

- acertos cegos antes e depois;
- ganho em pontos percentuais;
- desempenho por habilidade;
- fase escolhida pelo conjunto de desenvolvimento;
- caminho do checkpoint final;
- SHA256;
- status do gate técnico.

Mesmo que o gate passe, `ativo_no_chat=false` permanece.
